# Pertemuan 6 — Decrease & Conquer dan Struktur Data Efisien
Analisis teramortisasi, kompleksitas operasi struktur data Python, topological sort, dan **[Modern]** struktur data probabilistik (Bloom filter, Count-Min Sketch).

In [ ]:
import time, math, random, heapq, hashlib
from collections import defaultdict, deque, Counter
import numpy as np
import matplotlib.pyplot as plt
random.seed(11)

## 1. Operasi `in`: list O(n) vs set/dict O(1) rata-rata

In [ ]:
def tm(f):
    t = time.perf_counter(); f(); return time.perf_counter() - t
print(f"{'n':>8} {'list':>10} {'set':>10} {'dict':>10}   (1000 pencarian, detik)")
for n in [10**3, 10**4, 10**5]:
    L = list(range(n)); S = set(L); D = dict.fromkeys(L)
    q = [random.randrange(2 * n) for _ in range(1000)]
    print(f"{n:>8} {tm(lambda: [x in L for x in q]):>10.5f} {tm(lambda: [x in S for x in q]):>10.5f} {tm(lambda: [x in D for x in q]):>10.5f}")

## 2. Analisis teramortisasi: dynamic array
Strategi *doubling* memberi O(1) amortized per `append`; strategi "tambah 1 slot" memberi O(n).

In [ ]:
def total_copies(n, grow):
    cap, size, copies = 1, 0, 0
    for _ in range(n):
        if size == cap: copies += size; cap = grow(cap)
        size += 1
    return copies
print(f"{'n':>8} {'doubling':>10} {'+1 slot':>10}   salinan per append")
for n in [10**3, 10**4, 10**5]:
    d, p = total_copies(n, lambda c: 2 * c), total_copies(n, lambda c: c + 1)
    print(f"{n:>8} {d:>10} {p:>10}   {d/n:.2f} vs {p/n:.1f}")

## 3. Heap: top-k dan heapify

In [ ]:
n, k = 10**6, 10
data = [random.random() for _ in range(n)]
print("heapq.nlargest(k):", round(tm(lambda: heapq.nlargest(k, data)), 3), "s  ~ O(n log k)")
print("sorted()[-k:]    :", round(tm(lambda: sorted(data)[-k:]), 3), "s  ~ O(n log n)")
h = data[:]; print("heapify          :", round(tm(lambda: heapq.heapify(h)), 3), "s  ~ O(n)")

## 4. Topological Sort (Kahn) — decrease and conquer, O(V+E)

In [ ]:
def topo_sort(graph):
    indeg = defaultdict(int)
    for u in graph:
        indeg.setdefault(u, 0)
        for v in graph[u]: indeg[v] += 1
    q, order = deque(u for u in indeg if indeg[u] == 0), []
    while q:
        u = q.popleft(); order.append(u)
        for v in graph.get(u, []):
            indeg[v] -= 1
            if indeg[v] == 0: q.append(v)
    return order if len(order) == len(indeg) else None      # None → ada siklus
courses = {"Algo Dasar": ["Struktur Data"], "Struktur Data": ["Kompleksitas", "Basis Data"],
           "Matematika Diskrit": ["Kompleksitas"], "Kompleksitas": ["AI"], "Basis Data": ["AI"]}
print(topo_sort(courses)); print("Dengan siklus:", topo_sort({"a": ["b"], "b": ["a"]}))

## 5. [Modern] Bloom filter — memori kecil, ada *false positive*, tanpa *false negative*
Teori: FPR ≈ (1 − e^(−kn/m))ᵏ.

In [ ]:
class Bloom:
    def __init__(self, m, k): self.m, self.k, self.bits = m, k, bytearray(m)
    def _idx(self, item):
        h = hashlib.blake2b(str(item).encode(), digest_size=16).digest()
        h1, h2 = int.from_bytes(h[:8], "big"), int.from_bytes(h[8:], "big") | 1
        return [(h1 + i * h2) % self.m for i in range(self.k)]      # double hashing
    def add(self, item):
        for i in self._idx(item): self.bits[i] = 1
    def __contains__(self, item): return all(self.bits[i] for i in self._idx(item))

n = 10000
for bits_per_item in [4, 8, 12]:
    m = bits_per_item * n; k = max(1, round(bits_per_item * math.log(2)))
    bf = Bloom(m, k)
    for i in range(n): bf.add(f"item{i}")
    assert all(f"item{i}" in bf for i in range(n))                  # tidak ada false negative
    fp = sum(f"other{i}" in bf for i in range(20000)) / 20000
    print(f"{bits_per_item:>2} bit/item, k={k}: FPR terukur={fp:.4f}  teori={(1-math.exp(-k*n/m))**k:.4f}")

## 6. [Modern] Count-Min Sketch — estimasi frekuensi di *stream*
Estimasi tidak pernah di bawah nilai sebenarnya; galat ≤ ε·N dengan peluang ≥ 1−δ bila w = ⌈e/ε⌉, d = ⌈ln(1/δ)⌉.

In [ ]:
class CMS:
    def __init__(self, w, d): self.w, self.d, self.t = w, d, np.zeros((d, w), dtype=np.int64)
    def _h(self, x, i): return int.from_bytes(hashlib.blake2b(str(x).encode(), digest_size=8, salt=bytes([i])*8).digest(), "big") % self.w
    def add(self, x):
        for i in range(self.d): self.t[i, self._h(x, i)] += 1
    def est(self, x): return int(min(self.t[i, self._h(x, i)] for i in range(self.d)))

stream = [int(random.paretovariate(1.2)) for _ in range(50000)]
true = Counter(stream); N = len(stream)
cms = CMS(w=543, d=5)                       # ε≈0.005, δ≈0.007
for x in stream: cms.add(x)
top = [x for x, _ in true.most_common(5)]
print("item   benar  estimasi"); [print(f"{x:>5} {true[x]:>6} {cms.est(x):>9}") for x in top]
print("galat maksimum:", max(cms.est(x) - true[x] for x in true), " batas ε·N ≈", round(math.e / 543 * N))
print(f"memori CMS={cms.t.nbytes/1024:.1f} KiB vs Counter {len(true)} kunci")

## 7. Latihan / Tugas
1. Ukur operasi `list.insert(0, x)` vs `collections.deque.appendleft(x)` untuk n = 10³…10⁵ dan jelaskan kompleksitasnya.
2. Buktikan amortized O(1) dengan *metode akuntansi* (berapa "koin" per append?).
3. Implementasikan BFS dengan `deque` dan bandingkan dengan `list.pop(0)`.
4. **Tugas:** studi kasus — pilih struktur data (set / heap / dict / Bloom) untuk mengoptimalkan “deteksi duplikat pada 10⁷ URL” dan tunjukkan trade-off waktu, memori, dan akurasi.

In [ ]:
def bfs_deque(graph, s):
    pass  # TODO